In [14]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
DATA_YAML = DATA_DIR / "project_data.yaml"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_DIR)
print("YAML:", DATA_YAML)
print("YAML exists:", DATA_YAML.exists())

Project root: /Users/priyansh/Downloads/airport-baggage-screening
Dataset: /Users/priyansh/Downloads/airport-baggage-screening/data
YAML: /Users/priyansh/Downloads/airport-baggage-screening/data/project_data.yaml
YAML exists: True


In [15]:
print(DATA_YAML.read_text())

names:
- Baton
- Pliers
- Hammer
- Powerbank
- Scissors
- Wrench
- Gun
- Bullet
- Sprayer
- HandCuffs
- Knife
- Lighter
nc: 12
path: /Users/priyansh/Downloads/airport-baggage-screening/data
val: images/val
test: images/test
train: images/train



In [16]:
model = YOLO("yolo26n.pt")

In [17]:
model = YOLO("yolo26n.pt")

print("YOLO model loaded successfully.")

YOLO model loaded successfully.


In [18]:
print(model)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=

In [19]:
results = model.train(
    data=str(DATA_YAML),
    epochs=1,
    imgsz=640,
    batch=4,
    device="mps",
    workers=0,
    project=str(PROJECT_ROOT / "results"),
    name="sanity_check",
    exist_ok=True
)

Ultralytics 8.4.118 🚀 Python-3.13.5 torch-2.13.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/priyansh/Downloads/airport-baggage-screening/data/project_data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sanity_

/Users/priyansh/Downloads/airport-baggage-screening/venv/lib/python3.13/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/1      1.64G      1.464      5.583    0.01669          9        640: 100% ━━━━━━━━━━━━ 2746/2746 3.9it/s 11:480.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 172/172 4.4it/s 38.8s0.3ss
                   all       1373       1794      0.275      0.331      0.231      0.149

1 epochs completed in 0.209 hours.
Optimizer stripped from /Users/priyansh/Downloads/airport-baggage-screening/results/sanity_check/weights/last.pt, 5.4MB
Optimizer stripped from /Users/priyansh/Downloads/airport-baggage-screening/results/sanity_check/weights/best.pt, 5.4MB

Validating /Users/priyansh/Downloads/airport-baggage-screening/results/sanity_check/weights/best.pt...
Ultralytics 8.4.118 🚀 Python-3.13.5 torch-2.13.0 MPS (Apple M4)
YOLO26n summary (fused): 122 layers, 2,377,176 parameters, 0 gradients, 5.3 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 172/172 6.8it/s

In [20]:
best_model = YOLO(
    str(PROJECT_ROOT / "results" / "sanity_check" / "weights" / "best.pt")
)

print("Sanity-check model loaded.")

Sanity-check model loaded.


In [21]:
test_images = sorted(
    (DATA_DIR / "images" / "test").glob("*.jpg")
)

sample_test_image = test_images[0]

print("Test image:", sample_test_image)

Test image: /Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_00024.jpg


In [22]:
predictions = best_model.predict(
    source=str(sample_test_image),
    device="mps",
    conf=0.25,
    save=True
)


image 1/1 /Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_00024.jpg: 512x640 1 Baton, 19.8ms
Speed: 2.6ms preprocess, 19.8ms inference, 11.7ms postprocess per image at shape (1, 3, 512, 640)
Results saved to /Users/priyansh/runs/detect/predict-2


In [23]:
from IPython.display import Image, display

In [24]:
results = model.train(
    data=str(DATA_YAML),
    epochs=1,
    imgsz=640,
    batch=4,
    device="mps",
    workers=0,
    project=str(PROJECT_ROOT / "results"),
    name="sanity_check",
    exist_ok=True
)

Ultralytics 8.4.118 🚀 Python-3.13.5 torch-2.13.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/priyansh/Downloads/airport-baggage-screening/data/project_data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/Users/priyansh/Downloads/airport-baggage-screening/results/sanity_ch

/Users/priyansh/Downloads/airport-baggage-screening/venv/lib/python3.13/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/1      1.85G      1.419      3.279     0.0159          9        640: 100% ━━━━━━━━━━━━ 2746/2746 3.5it/s 12:590.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 172/172 4.0it/s 43.4s0.3ss
                   all       1373       1794      0.483        0.4      0.415      0.284

1 epochs completed in 0.230 hours.
Optimizer stripped from /Users/priyansh/Downloads/airport-baggage-screening/results/sanity_check/weights/last.pt, 5.4MB
Optimizer stripped from /Users/priyansh/Downloads/airport-baggage-screening/results/sanity_check/weights/best.pt, 5.4MB

Validating /Users/priyansh/Downloads/airport-baggage-screening/results/sanity_check/weights/best.pt...
Ultralytics 8.4.118 🚀 Python-3.13.5 torch-2.13.0 MPS (Apple M4)
YOLO26n summary (fused): 122 layers, 2,377,176 parameters, 0 gradients, 5.3 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 172/172 6.8it/s

# Baseline YOLO Training

The baseline model is trained using the original X-ray dataset without
additional preprocessing or data augmentation.

This model establishes the reference performance against which the
preprocessed and augmented models will be compared.

In [25]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_YAML = PROJECT_ROOT / "data" / "project_data.yaml"

model = YOLO("yolo26n.pt")

print("Baseline model loaded.")
print("Dataset:", DATA_YAML)

Baseline model loaded.
Dataset: /Users/priyansh/Downloads/airport-baggage-screening/data/project_data.yaml


In [26]:
baseline_results = model.train(
    data=str(DATA_YAML),

    epochs=30,
    imgsz=640,

    batch=8,
    device="mps",
    workers=0,

    # Disable augmentation for controlled baseline
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,

    fliplr=0.0,
    flipud=0.0,

    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,

    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    project=str(PROJECT_ROOT / "results"),
    name="baseline_yolo26n",
    exist_ok=True,

    plots=True
)

Ultralytics 8.4.118 🚀 Python-3.13.5 torch-2.13.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/priyansh/Downloads/airport-baggage-screening/data/project_data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=baseline

/Users/priyansh/Downloads/airport-baggage-screening/venv/lib/python3.13/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      3.83G       1.21      7.897    0.01705          8        640: 100% ━━━━━━━━━━━━ 1373/1373 1.9it/s 11:600.8ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 2.6it/s 32.5s0.3ss
                   all       1373       1794      0.385      0.325      0.303      0.206

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       2/30      3.72G      1.679      4.741    0.02807         10        640: 0% ──────────── 0/1373  1.1s

/Users/priyansh/Downloads/airport-baggage-screening/venv/lib/python3.13/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30       3.7G      1.183      3.197    0.01606         11        640: 100% ━━━━━━━━━━━━ 1373/1373 2.0it/s 11:350.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 2.5it/s 33.8s0.4ss
                   all       1373       1794      0.614      0.498      0.547      0.393

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/30       3.7G     0.7392      1.314    0.01356          8        640: 0% ──────────── 0/1373  1.0s

...

      28/30      3.67G     0.2721     0.1641   0.003104          6        640: 100% ━━━━━━━━━━━━ 1373/1373 2.0it/s 11:400.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 2.7it/s 32.4s0.3ss
                   all       1373       1794      0.862      0.744      0.798      0.627

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      29/30       3.7G     0.3476     0.3956   0.005469         14        640: 0% ──────────── 0/1373  0.8s

/Users/priyansh/Downloads/airport-baggage-screening/venv/lib/python3.13/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      3.69G     0.2613     0.1562   0.002996          9        640: 100% ━━━━━━━━━━━━ 1373/1373 1.9it/s 11:490.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 2.7it/s 32.2s0.3ss
                   all       1373       1794      0.869      0.742      0.796      0.627

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      30/30      3.67G     0.3915     0.2303   0.003348         10        640: 0% ──────────── 0/1373  0.7s

/Users/priyansh/Downloads/airport-baggage-screening/venv/lib/python3.13/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      3.71G     0.2531     0.1529   0.002891         13        640: 100% ━━━━━━━━━━━━ 1373/1373 1.9it/s 12:160.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 2.6it/s 32.7s0.3ss
                   all       1373       1794      0.886       0.73      0.798      0.629

30 epochs completed in 6.557 hours.
Optimizer stripped from /Users/priyansh/Downloads/airport-baggage-screening/results/baseline_yolo26n/weights/last.pt, 5.4MB
Optimizer stripped from /Users/priyansh/Downloads/airport-baggage-screening/results/baseline_yolo26n/weights/best.pt, 5.4MB

Validating /Users/priyansh/Downloads/airport-baggage-screening/results/baseline_yolo26n/weights/best.pt...
Ultralytics 8.4.118 🚀 Python-3.13.5 torch-2.13.0 MPS (Apple M4)
YOLO26n summary (fused): 122 layers, 2,377,176 parameters, 0 gradients, 5.3 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/8